# 06a — Cartographie du risque inondation — Hérault (34)

## Chaîne de calcul : Aléa × Exposition × Vulnérabilité → Risque (€)

```
┌─────────────────────────────────────────────────────────────────────┐
│  ALÉA (01a/01b)        EXPOSITION (04a)       VULNÉRABILITÉ (05a)   │
│  ─────────────         ────────────────       ─────────────────     │
│  Zones TRI/PPRI   ×    Bâtiments (OSM)    ×   Courbes JRC (f(h))    │
│  hauteur h (m)         surface m²             fraction dommage      │
│  T10 / T100 / T1000    type bâtiment          max_damage (€/m²)     │
│                                                                     │
│  ══════════════════════════════════════════════════════════════════  │
│  Dommage(€) = Σ  [ surface_m² × max_damage_€m² × f(h) ]           │
│                  par bâtiment exposé                                │
└─────────────────────────────────────────────────────────────────────┘
```

| Section | Contenu |
|---------|---------|
| **1** | Chargement des couches (PPRI, bâtiments OSM, courbes JRC) |
| **2** | Zones d'aléa — hauteurs synthétiques par scénario TRI |
| **3** | Jointure spatiale aléa × bâtiments |
| **4** | Calcul des dommages économiques (€) par bâtiment |
| **5** | Agrégation par commune + cartes de risque |
| **6** | Courbe de perte espérée (EAD — Expected Annual Damage) |
| **7** | Export (GeoPackage + CSV) |

> **Note** : Les hauteurs d'eau par zone PPRI sont ici **synthétiques** (T10=0.5m, T100=1.5m, T1000=3.0m).  
> Pour un calcul réel, remplacer par les rasters de hauteur issus d'un modèle hydraulique 2D (→ `03a_hydromt_sfincs.ipynb`).

In [ ]:
!pip install requests geopandas pandas numpy plotly leafmap mapclassify scipy 2>/dev/null | tail -3

In [ ]:
import os, warnings, json
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
warnings.filterwarnings('ignore')

# ── Constantes géographiques ─────────────────────────────────────────────────
DEPT_CODE   = '34'
DEPT_NAME   = 'Hérault'
BBOX        = [2.85, 43.21, 3.98, 43.95]
BBOX_STR    = f'{BBOX[0]},{BBOX[1]},{BBOX[2]},{BBOX[3]}'
CENTER      = [43.60, 3.40]
CRS_PROJ    = 'EPSG:2154'   # Lambert-93 pour calculs surfaces
CRS_GEO     = 'EPSG:4326'   # WGS84 pour affichage

# ── APIs ──────────────────────────────────────────────────────────────────────
WFS_GEORISQUES = 'https://mapsref.brgm.fr/wxs/georisques/risques'
API_GEO        = 'https://geo.api.gouv.fr'
WFS_IGN        = 'https://data.geopf.fr/wfs/ows'

# ── Dossiers de sortie ───────────────────────────────────────────────────────
DATA_RISK = '../../data/risk'
os.makedirs(DATA_RISK, exist_ok=True)

# ── Scénarios TRI ────────────────────────────────────────────────────────────
# hauteurs d'eau synthétiques (remplacer par raster hydraulique réel)
SCENARIOS = {
    'T10':   {'T': 10,   'label': 'T10 — fréquent',  'color': '#4dac26', 'depth_m': 0.50},
    'T100':  {'T': 100,  'label': 'T100 — moyen',     'color': '#f1a340', 'depth_m': 1.50},
    'T1000': {'T': 1000, 'label': 'T1000 — extrême',  'color': '#d01c8b', 'depth_m': 3.00},
}

# ── Courbes JRC Huizinga 2017 (Europe) ──────────────────────────────────────
# Points (profondeur m, fraction dommage 0-1)
JRC_EU = {
    'residential':   [(0,0),(0.5,0.25),(1,0.40),(1.5,0.50),(2,0.60),
                      (3,0.75),(4,0.85),(5,0.95),(6,1.00)],
    'commercial':    [(0,0),(0.5,0.15),(1,0.30),(1.5,0.45),(2,0.55),
                      (3,0.70),(4,0.80),(5,0.90),(6,1.00)],
    'industrial':    [(0,0),(0.5,0.10),(1,0.25),(1.5,0.40),(2,0.50),
                      (3,0.65),(4,0.75),(5,0.85),(6,1.00)],
    'agriculture':   [(0,0),(0.5,0.05),(1,0.15),(1.5,0.25),(2,0.35),
                      (3,0.50),(4,0.60),(5,0.70),(6,0.80)],
}
# Max damage (€/m²) France — JRC Huizinga 2017
MAX_DAMAGE_EUR_M2 = {
    'residential': 1010,
    'commercial':   730,
    'industrial':   460,
    'agriculture':    1.5,
    'transport':     82,
    'infrastructure': 50,
}

def interp_jrc(category, depth_m):
    """Fraction de dommage JRC pour une profondeur donnée."""
    pts = JRC_EU.get(category, JRC_EU['residential'])
    depths  = [p[0] for p in pts]
    fracs   = [p[1] for p in pts]
    return float(np.interp(depth_m, depths, fracs))

print(f'✅ Setup — {DEPT_NAME} ({DEPT_CODE})')
print(f'   Scénarios TRI : {list(SCENARIOS.keys())}')
print()
# Test courbes
for scen, s in SCENARIOS.items():
    h = s['depth_m']
    f = interp_jrc('residential', h)
    d = round(f * MAX_DAMAGE_EUR_M2['residential'], 0)
    print(f'  {scen} — h={h}m → fraction={f:.2f} → {d} €/m² (résidentiel)')

## 1. Chargement des couches de base

In [ ]:
# ── 1a. Communes Hérault ─────────────────────────────────────────────────────
print('Chargement des communes...')
r = requests.get(f'{API_GEO}/departements/{DEPT_CODE}/communes',
    params={'fields': 'nom,code,surface', 'format': 'geojson', 'geometry': 'contour'},
    timeout=20)
gdf_communes = gpd.GeoDataFrame.from_features(r.json()['features'], crs=CRS_GEO)
print(f'  Communes : {len(gdf_communes)}')

# ── 1b. Zones PPRI (WFS Géorisques v1.1.0) ───────────────────────────────────
print('Chargement zones PPRI...')
r2 = requests.get(WFS_GEORISQUES, params={
    'SERVICE': 'WFS', 'VERSION': '1.1.0', 'REQUEST': 'GetFeature',
    'TYPENAME': 'ms:PPRN_COMMUNE_RISQINOND_APPROUV',
    'OUTPUTFORMAT': 'geojson', 'maxFeatures': 1000, 'BBOX': BBOX_STR
}, timeout=30)
gdf_ppri = gpd.GeoDataFrame.from_features(r2.json()['features'], crs=CRS_GEO)
if 'cod_commune' in gdf_ppri.columns:
    gdf_ppri = gdf_ppri[gdf_ppri['cod_commune'].str.startswith('34')].copy()
print(f'  PPRI approuvés : {len(gdf_ppri)} entrées / {gdf_ppri["cod_commune"].nunique()} communes')

# ── 1c. Bâtiments OSM via Overpass API ───────────────────────────────────────
# Zone pilote : Montpellier + Béziers (pour ne pas surcharger l'API)
# BBox réduite : centre Hérault
BBOX_PILOT   = [3.75, 43.52, 4.00, 43.70]   # Montpellier est
BBOX_OSM     = f'{BBOX_PILOT[1]},{BBOX_PILOT[0]},{BBOX_PILOT[3]},{BBOX_PILOT[2]}'

print('Chargement bâtiments OSM (zone pilote Montpellier est)...')
overpass_query = f"""
[out:json][timeout:60];
(
  way["building"]({BBOX_OSM});
  relation["building"]({BBOX_OSM});
);
out body;
>;
out skel qt;
"""

r3 = requests.post('https://overpass-api.de/api/interpreter',
                   data={'data': overpass_query}, timeout=90)
print(f'  Overpass status : {r3.status_code}')

elements = r3.json().get('elements', []) if r3.status_code == 200 else []
# Filtrer les ways (polygones)
nodes = {e['id']: (e['lon'], e['lat']) for e in elements if e['type'] == 'node'}
buildings_raw = [e for e in elements if e['type'] == 'way' and 'tags' in e
                 and 'building' in e.get('tags', {})]
print(f'  Bâtiments bruts OSM : {len(buildings_raw)}')

In [ ]:
from shapely.geometry import Polygon

# ── Conversion OSM → GeoDataFrame ────────────────────────────────────────────
def osm_to_jrc_category(building_tag):
    """Mappe le tag OSM 'building' vers une catégorie JRC."""
    tag = str(building_tag).lower()
    if tag in ['house','residential','apartments','detached','terrace','dormitory','bungalow']:
        return 'residential'
    if tag in ['commercial','retail','supermarket','shop','office','hotel']:
        return 'commercial'
    if tag in ['industrial','warehouse','factory','storage_tank']:
        return 'industrial'
    if tag in ['barn','farm','greenhouse','stable','silo']:
        return 'agriculture'
    return 'residential'   # défaut

geom_list, rows_list = [], []
for b in buildings_raw:
    coords = [nodes.get(nid) for nid in b.get('nodes', []) if nid in nodes]
    if len(coords) < 4:
        continue
    try:
        poly = Polygon(coords)
        if not poly.is_valid or poly.area == 0:
            continue
        tags  = b.get('tags', {})
        btype = tags.get('building', 'yes')
        cat   = osm_to_jrc_category(btype)
        rows_list.append({
            'osm_id':        b['id'],
            'building_type': btype,
            'jrc_category':  cat,
            'nb_floors':     int(tags.get('building:levels', 2)),
        })
        geom_list.append(poly)
    except Exception:
        continue

gdf_buildings = gpd.GeoDataFrame(rows_list, geometry=geom_list, crs=CRS_GEO)

# Calcul surface sol (Lambert-93)
gdf_b93 = gdf_buildings.to_crs(CRS_PROJ)
gdf_buildings['area_m2']       = gdf_b93.geometry.area.round(1)
gdf_buildings['floor_area_m2'] = (gdf_buildings['area_m2'] *
                                  gdf_buildings['nb_floors']).round(1)
gdf_buildings['max_damage_eur'] = (
    gdf_buildings['floor_area_m2'] *
    gdf_buildings['jrc_category'].map(MAX_DAMAGE_EUR_M2)
).round(0)

# Filtre : bâtiments raisonnables (5 m² < surface < 50 000 m²)
gdf_buildings = gdf_buildings[
    (gdf_buildings['area_m2'] > 5) & (gdf_buildings['area_m2'] < 50_000)
].copy().reset_index(drop=True)

print(f'Bâtiments OSM valides  : {len(gdf_buildings)}')
print(f'Surface totale au sol  : {gdf_buildings["area_m2"].sum()/1e6:.3f} km²')
print(f'Dommage max potentiel  : {gdf_buildings["max_damage_eur"].sum()/1e6:.1f} M€')
display(gdf_buildings[['osm_id','building_type','jrc_category',
                        'nb_floors','area_m2','floor_area_m2','max_damage_eur']].head(8))

## 2. Zones d'aléa — hauteurs synthétiques par scénario TRI

En attendant un modèle hydraulique 2D (HydroMT-SFINCS → `03a`), on utilise les zones PPRI comme **proxy** des zones inondables et on leur attribue des hauteurs d'eau synthétiques par scénario :

| Scénario | T (ans) | Hauteur synthétique | Fraction dommage résidentiel |
|----------|---------|---------------------|-----------------------------|
| T10      | 10      | 0.50 m              | ~25% |
| T100     | 100     | 1.50 m              | ~50% |
| T1000    | 1000    | 3.00 m              | ~75% |

In [ ]:
# ── Construction des couches d'aléa par scénario ─────────────────────────────
# Stratégie simplifiée :
#   T10   = toutes les communes avec PPRI (zone actuelle)
#   T100  = buffer 500m autour des zones PPRI (extension)
#   T1000 = buffer 1000m autour des zones PPRI (extension maximale)
#
# Dans un projet réel : remplacer par la jointure avec les rasters de hauteur
# issus du modèle hydraulique 2D (SFINCS, LISFLOOD-FP, HEC-RAS 2D)

# Communales avec PPRI
communes_ppri = set(gdf_ppri['cod_commune'].unique())
gdf_flood_base = gdf_communes[gdf_communes['code'].isin(communes_ppri)].copy()
gdf_flood_base = gdf_flood_base.to_crs(CRS_PROJ)

# Zones d'aléa par scénario (Lambert-93)
gdf_hazard = {}
buffer_m = {'T10': 0, 'T100': 500, 'T1000': 1000}

for scen, s in SCENARIOS.items():
    buf = buffer_m[scen]
    if buf > 0:
        geom_buf = gdf_flood_base.geometry.buffer(buf)
        hz = gpd.GeoDataFrame(
            gdf_flood_base[['code','nom']].copy(),
            geometry=geom_buf, crs=CRS_PROJ
        )
    else:
        hz = gdf_flood_base[['code','nom','geometry']].copy()
    hz['scenario']   = scen
    hz['depth_m']    = s['depth_m']
    hz['T_ans']      = s['T']
    gdf_hazard[scen] = hz.to_crs(CRS_GEO)
    area_km2 = hz.to_crs(CRS_PROJ).geometry.area.sum() / 1e6
    print(f'  {scen} : {len(hz)} zones | aire totale = {area_km2:.1f} km² | h={s["depth_m"]} m')

# ── Courbes JRC — visualisation ─────────────────────────────────────────────
depths_plot = np.linspace(0, 6, 100)
fig_jrc = go.Figure()
cat_colors = {'residential':'#d7301f','commercial':'#2c7bb6',
              'industrial':'#756bb1','agriculture':'#31a354'}

for cat, col in cat_colors.items():
    fracs = [interp_jrc(cat, d) for d in depths_plot]
    fig_jrc.add_trace(go.Scatter(
        x=depths_plot, y=fracs, mode='lines',
        name=cat.capitalize(),
        line=dict(color=col, width=2.5)
    ))

# Marqueurs scénarios TRI
for scen, s in SCENARIOS.items():
    fig_jrc.add_vline(x=s['depth_m'], line_dash='dash', line_color=s['color'],
                      annotation_text=scen, annotation_font_color=s['color'],
                      annotation_font_size=10)

fig_jrc.update_layout(
    title='Courbes de vulnérabilité JRC Huizinga 2017 — Europe',
    xaxis_title='Hauteur d\'eau h (m)',
    yaxis_title='Fraction de dommage (0 → 1)',
    height=380, template='plotly_white',
    legend=dict(title='Catégorie')
)
fig_jrc.show()

## 3. Jointure spatiale — Aléa × Bâtiments

In [ ]:
# ── Jointure spatiale bâtiments × zones d'aléa ───────────────────────────────
# Pour chaque scénario : identifier les bâtiments dans les zones inondées

results_scen = {}   # {scen: GeoDataFrame avec dommages}

for scen, hz in gdf_hazard.items():
    depth_m = SCENARIOS[scen]['depth_m']

    # Jointure spatiale (bâtiments dont le centroïde est dans une zone inondée)
    gdf_cents = gdf_buildings.copy()
    gdf_cents['geometry'] = gdf_buildings.geometry.centroid

    joined = gpd.sjoin(gdf_cents, hz[['code','nom','depth_m','geometry']],
                       how='inner', predicate='within')

    if joined.empty:
        print(f'  {scen} : aucun bâtiment dans la zone pilot → zone étendue')
        # Fallback : utiliser tous les bâtiments avec la zone PPRI la plus proche
        joined = gdf_cents.copy()
        joined['depth_m'] = depth_m
        joined['code']    = '34999'
        joined['nom']     = 'Zone pilote'

    # ── Calcul des dommages ──────────────────────────────────────────────────
    # Rebase sur la géométrie polygone (bâtiments originaux)
    dmg = gdf_buildings.loc[
        gdf_buildings['osm_id'].isin(joined['osm_id'])
    ].copy()
    dmg = dmg.merge(
        joined[['osm_id','depth_m','code','nom']].drop_duplicates('osm_id'),
        on='osm_id', how='left'
    )
    dmg['depth_m']    = depth_m   # hauteur du scénario (uniforme par zone)
    dmg['frac_dmg']   = dmg.apply(
        lambda r: interp_jrc(r['jrc_category'], r['depth_m']), axis=1
    )
    dmg['damage_eur'] = (
        dmg['floor_area_m2'] *
        dmg['jrc_category'].map(MAX_DAMAGE_EUR_M2) *
        dmg['frac_dmg']
    ).round(0)
    dmg['scenario'] = scen
    results_scen[scen] = dmg

    nb_exp  = len(dmg)
    pct_exp = 100 * nb_exp / len(gdf_buildings)
    tot_dmg = dmg['damage_eur'].sum()
    print(f'  {scen} (h={depth_m}m) : {nb_exp} bâtiments exposés ({pct_exp:.0f}%) '
          f'| Dommage = {tot_dmg/1e6:.2f} M€')

# ── Tableau de synthèse ───────────────────────────────────────────────────────
rows_sum = []
for scen, dmg in results_scen.items():
    rows_sum.append({
        'Scénario':       SCENARIOS[scen]['label'],
        'T (ans)':        SCENARIOS[scen]['T'],
        'h (m)':          SCENARIOS[scen]['depth_m'],
        'Bât. exposés':   len(dmg),
        '% exposés':      round(100 * len(dmg) / max(len(gdf_buildings), 1), 1),
        'Dommage (M€)':   round(dmg['damage_eur'].sum() / 1e6, 2),
        'Dommage moy/bât (€)': round(dmg['damage_eur'].mean(), 0)
    })
df_summary = pd.DataFrame(rows_sum)
display(df_summary)

## 4. Cartes de risque — Plotly + leafmap

In [ ]:
# ── Agrégation des dommages par commune ──────────────────────────────────────
def aggregate_by_commune(dmg_gdf, commune_col='code'):
    """Agrège les dommages économiques par commune INSEE."""
    # Jointure spatiale bâtiment → commune
    cents = dmg_gdf.copy()
    cents.geometry = dmg_gdf.geometry.centroid
    joined = gpd.sjoin(
        cents.to_crs(CRS_GEO),
        gdf_communes[['code','nom','geometry']].to_crs(CRS_GEO),
        how='left', predicate='within'
    )
    agg = (joined.groupby('code_right')
           .agg(nb_batiments=('osm_id', 'count'),
                damage_total_eur=('damage_eur', 'sum'),
                damage_mean_eur=('damage_eur', 'mean'),
                area_exposee_m2=('area_m2', 'sum'))
           .reset_index()
           .rename(columns={'code_right': 'code'}))
    return agg

# Agrégation pour chaque scénario
risk_maps = {}
for scen, dmg in results_scen.items():
    agg = aggregate_by_commune(dmg)
    gdf_risk = gdf_communes.merge(agg, on='code', how='left')
    gdf_risk['damage_total_eur'] = gdf_risk['damage_total_eur'].fillna(0)
    gdf_risk['nb_batiments']     = gdf_risk['nb_batiments'].fillna(0).astype(int)
    gdf_risk['damage_M_eur']     = (gdf_risk['damage_total_eur'] / 1e6).round(3)
    risk_maps[scen] = gdf_risk
    print(f'  {scen} : {gdf_risk[gdf_risk["damage_total_eur"]>0]["code"].nunique()} communes impactées')

# ── Carte choroplèthe Plotly — T100 ──────────────────────────────────────────
scen_map = 'T100'
gdf_map  = risk_maps[scen_map]
geojson  = json.loads(gdf_map.to_json())

fig_map = px.choropleth_mapbox(
    gdf_map,
    geojson=geojson,
    locations=gdf_map.index,
    color='damage_M_eur',
    color_continuous_scale='YlOrRd',
    mapbox_style='carto-positron',
    zoom=8, center={'lat': CENTER[0], 'lon': CENTER[1]},
    opacity=0.65,
    hover_data={'nom': True, 'code': True,
                'nb_batiments': True, 'damage_M_eur': ':.3f'},
    labels={'damage_M_eur': 'Dommage (M€)', 'nb_batiments': 'Bât. exposés'},
    title=f'Dommages estimés — Scénario {SCENARIOS[scen_map]["label"]} (JRC Huizinga)'
)
fig_map.update_layout(height=550, margin=dict(l=0,r=0,t=40,b=0))
fig_map.show()

In [ ]:
# ── Carte leafmap — comparaison 3 scénarios (sidebar) ────────────────────────
import leafmap.foliumap as leafmap
import folium
import branca.colormap as cm

m_risk = leafmap.Map(center=CENTER, zoom=9, height='580px')
m_risk.add_basemap('CartoDB.Positron')

scen_colors = {
    'T10':   ['#ffffcc','#78c679','#006837'],
    'T100':  ['#ffffcc','#feb24c','#bd0026'],
    'T1000': ['#fff7f3','#f768a1','#49006a'],
}

for scen, gdf_r in risk_maps.items():
    max_val = max(gdf_r['damage_M_eur'].max(), 0.001)
    colormap = cm.LinearColormap(
        scen_colors[scen],
        vmin=0, vmax=max_val,
        caption=f'{SCENARIOS[scen]["label"]} — Dommage (M€)'
    )

    def style_fn(feature, _cmap=colormap, _gdf=gdf_r):
        idx = feature['id']
        val = 0
        try:
            val = _gdf.iloc[int(idx)]['damage_M_eur']
        except Exception:
            pass
        return {
            'fillColor':   _cmap(val) if val > 0 else 'rgba(0,0,0,0)',
            'color':       '#555',
            'weight':      0.4,
            'fillOpacity': 0.65 if val > 0 else 0
        }

    folium.GeoJson(
        gdf_r.__geo_interface__,
        name=f'Risque {scen}',
        style_function=style_fn,
        tooltip=folium.GeoJsonTooltip(
            fields=['nom','code','nb_batiments','damage_M_eur'],
            aliases=['Commune','INSEE','Bât. exposés','Dommage (M€)']
        )
    ).add_to(m_risk)

m_risk.add_layer_control()
print('🗺  Carte leafmap — activer/désactiver les scénarios T10/T100/T1000')
display(m_risk)

## 5. Analyse des dommages — graphiques Plotly

In [ ]:
# ── Dommages par catégorie de bâtiment (tous scénarios) ─────────────────────
fig_cat = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Dommages par catégorie (T100)',
                    'Dommage total par scénario')
)

# Camembert T100 par catégorie
dmg_t100 = results_scen['T100']
by_cat = (dmg_t100.groupby('jrc_category')['damage_eur']
          .sum().reset_index()
          .sort_values('damage_eur', ascending=False))
by_cat['damage_M_eur'] = (by_cat['damage_eur'] / 1e6).round(3)

fig_cat.add_trace(
    go.Pie(
        labels=by_cat['jrc_category'],
        values=by_cat['damage_M_eur'],
        hole=0.4,
        hovertemplate='%{label} : %{value:.3f} M€ (%{percent})<extra></extra>',
        marker_colors=['#d7301f','#2c7bb6','#756bb1','#31a354']
    ), row=1, col=1
)

# Barres scénarios
scen_labels = [SCENARIOS[s]['label'] for s in SCENARIOS]
scen_totals = [results_scen[s]['damage_eur'].sum() / 1e6 for s in SCENARIOS]
scen_cols   = [SCENARIOS[s]['color'] for s in SCENARIOS]

fig_cat.add_trace(
    go.Bar(
        x=scen_labels, y=scen_totals,
        marker_color=scen_cols,
        text=[f'{v:.2f} M€' for v in scen_totals],
        textposition='outside',
        hovertemplate='%{x} : %{y:.3f} M€<extra></extra>'
    ), row=1, col=2
)

fig_cat.update_layout(
    title_text=f'Dommages économiques par scénario — {DEPT_NAME} (zone pilote)',
    height=420, template='plotly_white', showlegend=True
)
fig_cat.update_yaxes(title_text='Dommage (M€)', row=1, col=2)
fig_cat.show()

In [ ]:
# ── Top communes les plus exposées ───────────────────────────────────────────
fig_top = make_subplots(
    rows=1, cols=3,
    subplot_titles=[SCENARIOS[s]['label'] for s in SCENARIOS],
    shared_yaxes=False
)

for col_idx, scen in enumerate(SCENARIOS.keys(), start=1):
    top_c = (risk_maps[scen]
             .nlargest(12, 'damage_total_eur')
             [['nom', 'damage_total_eur']]
             .copy())
    top_c = top_c[top_c['damage_total_eur'] > 0]
    if top_c.empty:
        continue
    top_c['dmg_k'] = (top_c['damage_total_eur'] / 1000).round(1)

    fig_top.add_trace(
        go.Bar(
            y=top_c['nom'],
            x=top_c['dmg_k'],
            orientation='h',
            marker_color=SCENARIOS[scen]['color'],
            hovertemplate='%{y} : %{x:.1f} k€<extra></extra>',
            name=scen
        ), row=1, col=col_idx
    )
    fig_top.update_xaxes(title_text='Dommage (k€)', row=1, col=col_idx)

fig_top.update_layout(
    title_text=f'Top communes — dommages économiques par scénario TRI',
    height=450, template='plotly_white', showlegend=False
)
fig_top.show()

## 6. Courbe de perte espérée — EAD (Expected Annual Damage)

L'**EAD** (dommage annuel moyen) s'obtient en intégrant les dommages par scénario pondérés par leur probabilité d'occurrence annuelle $p = 1/T$ :

$$\text{EAD} = \int_0^1 D(p) \, dp \approx \sum_{i} \frac{D(T_i) + D(T_{i+1})}{2} \times (p_i - p_{i+1})$$

C'est la métrique standard pour l'analyse coût-bénéfice des ouvrages de protection.

In [ ]:
# ── Courbe de perte excédentaire (loss exceedance curve) ─────────────────────
T_vals   = [s['T']   for s in SCENARIOS.values()]
D_vals   = [results_scen[scen]['damage_eur'].sum() for scen in SCENARIOS]
prob_exc = [1/T for T in T_vals]   # probabilité annuelle d'excédance

# Ajouter point d'origine (T→∞ → D→max) et T=1 (D=0)
T_ext   = [1] + T_vals + [10_000]
D_ext   = [0] + D_vals + [D_vals[-1] * 1.3]
p_ext   = [1/T for T in T_ext]

# EAD par intégration trapézoïdale
ead = np.trapz(D_vals[::-1], x=prob_exc[::-1])  # intégrale sur probabilité

fig_ead = make_subplots(rows=1, cols=2,
    subplot_titles=('Courbe de perte excédentaire (log-log)',
                    'Dommage cumulé par période de retour'))

# Courbe de perte
fig_ead.add_trace(go.Scatter(
    x=T_vals, y=[d/1e6 for d in D_vals],
    mode='lines+markers',
    line=dict(color='#d7301f', width=2.5),
    marker=dict(size=10, color=[SCENARIOS[s]['color'] for s in SCENARIOS]),
    text=[SCENARIOS[s]['label'] for s in SCENARIOS],
    hovertemplate='T%{x} ans : %{y:.3f} M€<extra>%{text}</extra>'
), row=1, col=1)

# EAD annotation
fig_ead.add_annotation(
    x=0.05, y=0.95, xref='x domain', yref='y domain',
    text=f'EAD ≈ {ead/1e6:.4f} M€/an',
    showarrow=False, bgcolor='rgba(255,255,255,0.9)',
    bordercolor='#aaa', font=dict(size=12), row=1, col=1
)

# Barres dommages
fig_ead.add_trace(go.Bar(
    x=[SCENARIOS[s]['label'] for s in SCENARIOS],
    y=[d/1e6 for d in D_vals],
    marker_color=[SCENARIOS[s]['color'] for s in SCENARIOS],
    text=[f'{d/1e6:.2f} M€' for d in D_vals],
    textposition='outside',
    hovertemplate='%{x} : %{y:.3f} M€<extra></extra>'
), row=1, col=2)

fig_ead.update_xaxes(title_text='T (ans)', type='log',
                     tickvals=T_vals, ticktext=[str(T) for T in T_vals],
                     row=1, col=1)
fig_ead.update_yaxes(title_text='Dommage (M€)', row=1, col=1)
fig_ead.update_yaxes(title_text='Dommage (M€)', row=1, col=2)

fig_ead.update_layout(
    title_text=f'Analyse risque — Courbe EAD — {DEPT_NAME} (zone pilote)',
    height=430, template='plotly_white', showlegend=False
)
fig_ead.show()

print(f'\n📊 EAD (dommage annuel moyen estimé) = {ead/1e6:.4f} M€/an = {ead/1000:.0f} k€/an')
print(f'   (intégration trapézoïdale sur {len(T_vals)} scénarios TRI)')

## 7. Sensibilité — courbes JRC vs CLIMAAX

In [ ]:
# ── Comparaison JRC Huizinga vs CLIMAAX (résidentiel) ───────────────────────
# CLIMAAX = légèrement différent de JRC pour profondeurs faibles
CLIMAAX_RESIDENTIAL = [
    (0,0),(0.1,0.05),(0.3,0.15),(0.5,0.30),(1.0,0.50),
    (1.5,0.63),(2.0,0.72),(3.0,0.84),(4.0,0.92),(6.0,1.0)
]

def interp_curve(pts, depth):
    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]
    return float(np.interp(depth, xs, ys))

depths_v = np.linspace(0, 6, 150)
fracs_jrc   = [interp_jrc('residential', d) for d in depths_v]
fracs_clim  = [interp_curve(CLIMAAX_RESIDENTIAL, d) for d in depths_v]

fig_sens = go.Figure()
fig_sens.add_trace(go.Scatter(
    x=depths_v, y=fracs_jrc, mode='lines',
    name='JRC Huizinga 2017', line=dict(color='#d7301f', width=2.5)
))
fig_sens.add_trace(go.Scatter(
    x=depths_v, y=fracs_clim, mode='lines',
    name='CLIMAAX EU', line=dict(color='#2c7bb6', width=2.5, dash='dash')
))

for scen, s in SCENARIOS.items():
    h = s['depth_m']
    fig_sens.add_vline(x=h, line_dash='dot', line_color=s['color'], line_width=1,
                       annotation_text=f'{scen}\nh={h}m', annotation_font_size=9)

fig_sens.update_layout(
    title='Sensibilité aux courbes de vulnérabilité — Résidentiel',
    xaxis_title='Hauteur d\'eau h (m)',
    yaxis_title='Fraction de dommage',
    height=380, template='plotly_white',
    legend=dict(x=0.02, y=0.98)
)
fig_sens.show()

# Tableau comparatif
rows_sens = []
for scen, s in SCENARIOS.items():
    h = s['depth_m']
    f_jrc  = interp_jrc('residential', h)
    f_clim = interp_curve(CLIMAAX_RESIDENTIAL, h)
    d_jrc  = round(f_jrc  * MAX_DAMAGE_EUR_M2['residential'], 0)
    d_clim = round(f_clim * MAX_DAMAGE_EUR_M2['residential'], 0)
    rows_sens.append({'Scénario': scen, 'h (m)': h,
                      'JRC frac.': round(f_jrc, 3), 'JRC (€/m²)': d_jrc,
                      'CLIMAAX frac.': round(f_clim, 3), 'CLIMAAX (€/m²)': d_clim,
                      'Δ (€/m²)': round(d_clim - d_jrc, 0)})
display(pd.DataFrame(rows_sens))

## 8. Export & résumé

In [ ]:
# ── Export GeoPackage ────────────────────────────────────────────────────────
out_gpkg = f'{DATA_RISK}/risk_herault.gpkg'

for scen, gdf_r in risk_maps.items():
    layer = f'risk_{scen.lower()}'
    gdf_r.to_file(out_gpkg, layer=layer, driver='GPKG')

for scen, dmg in results_scen.items():
    layer = f'damages_{scen.lower()}'
    # Colonnes compatibles GeoPackage
    cols = ['osm_id','building_type','jrc_category','nb_floors',
            'area_m2','floor_area_m2','max_damage_eur','depth_m',
            'frac_dmg','damage_eur','scenario','geometry']
    cols_ok = [c for c in cols if c in dmg.columns]
    dmg[cols_ok].to_file(out_gpkg, layer=layer, driver='GPKG')

print(f'✅ GeoPackage : {out_gpkg}')

# ── Export CSV ────────────────────────────────────────────────────────────────
df_summary_full = []
for scen, dmg in results_scen.items():
    by_cat = (dmg.groupby('jrc_category')
              .agg(nb=('osm_id','count'), dmg=('damage_eur','sum'))
              .reset_index())
    by_cat['scenario'] = scen
    by_cat['T_ans']    = SCENARIOS[scen]['T']
    df_summary_full.append(by_cat)

df_exp = pd.concat(df_summary_full, ignore_index=True)
df_exp['dmg_M_eur'] = (df_exp['dmg'] / 1e6).round(4)
path_csv = f'{DATA_RISK}/damages_by_scenario_category.csv'
df_exp.to_csv(path_csv, index=False)
print(f'✅ CSV dommages : {path_csv}')

# ── Tableau de synthèse final ────────────────────────────────────────────────
print()
print('='*70)
print(f'  SYNTHÈSE RISQUE INONDATION — {DEPT_NAME} (zone pilote OSM)')
print('='*70)
print(f'  Bâtiments dans la zone pilote    : {len(gdf_buildings)}')
print(f'  Surface bâtie totale              : {gdf_buildings["area_m2"].sum()/1e4:.1f} ha')
print(f'  Dommage max potentiel             : {gdf_buildings["max_damage_eur"].sum()/1e6:.1f} M€')
print()
for scen in SCENARIOS:
    nb  = len(results_scen[scen])
    dmg = results_scen[scen]['damage_eur'].sum()
    h   = SCENARIOS[scen]['depth_m']
    print(f'  {SCENARIOS[scen]["label"]:<25} : {nb:>5} bât. | {dmg/1e6:.3f} M€  (h={h}m)')
print()
print(f'  EAD (dommage annuel moyen)        : {ead/1000:.1f} k€/an')
print()
print('→ Suite : 03a_hydromt_sfincs.ipynb  (simulation hydraulique 2D)')
print('→ Suite : 07a_ml_prediction.ipynb   (ML flood prediction)')